# 10 — Regression Dixon-Coles

Replaces per-team `alpha/beta` free parameters with linear functions of team features. Goal: cold-start friendly for 2026 squad rebuilds, and re-use features that already proved themselves in notebook 09 (elo, log_value, fifa_points).

Backtest: walk-forward WC 2010-2022, vs plain DC.

In [1]:
import sys
sys.path.append('..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from sklearn.metrics import log_loss
from src.elo import EloModel
from src.draw_model import match_outcome
from src.dixon_coles import DixonColesModel
from src.regression_dc import RegressionDixonColesModel

## Build per-team features per match

Reg-DC needs `home_{feat}` and `away_{feat}` for each feature. We'll use what notebook 09 found useful: elo (pre-match), log10(squad_value+floor), fifa_points.

In [2]:
df_all = pd.read_csv('../data/processed/matches_competitive.csv', parse_dates=['date']).dropna(subset=['home_score','away_score'])
df_sv = pd.read_csv('../data/processed/matches_with_squad_value.csv', parse_dates=['date'])
elo = EloModel()
elo_enriched = elo.fit(df_all)
df = df_sv.merge(elo_enriched[['date','home_team','away_team','home_elo_pre','away_elo_pre']],
                  on=['date','home_team','away_team'], how='left').dropna(subset=['home_elo_pre','away_elo_pre'])

# Per-team features (NOT diffs). Reg-DC ingests team features individually.
FLOOR = 1e5
df['home_elo']   = df['home_elo_pre']   / 100.0  # rescale to ~O(1)
df['away_elo']   = df['away_elo_pre']   / 100.0
df['home_logv']  = np.log10(df['home_top_n_value_eur'].clip(lower=FLOOR))
df['away_logv']  = np.log10(df['away_top_n_value_eur'].clip(lower=FLOOR))
df['home_fpts']  = df['home_fifa_points'].fillna(0) / 1000.0
df['away_fpts']  = df['away_fifa_points'].fillna(0) / 1000.0

FEATURE_COLS = ['elo', 'logv', 'fpts']
needed = ['home_score','away_score'] + [f'{s}_{c}' for s in ('home','away') for c in FEATURE_COLS]
valid = df.dropna(subset=needed)
valid = valid[(valid['home_top_n_value_eur'] > FLOOR) & (valid['away_top_n_value_eur'] > FLOOR)].copy()
valid['outcome'] = valid.apply(lambda r: match_outcome(r['home_score'], r['away_score']), axis=1)
print(f'Training-eligible matches: {len(valid):,}')
print(valid[[f'{s}_{c}' for s in ('home','away') for c in FEATURE_COLS]].describe().round(3))

Training-eligible matches: 7,673
       home_elo  home_logv  home_fpts  away_elo  away_logv  away_fpts
count  7673.000   7673.000   7673.000  7673.000   7673.000   7673.000
mean     15.590      7.454      0.939    15.519      7.407      0.926
std       1.361      0.931      0.488     1.338      0.925      0.486
min      10.724      5.097      0.000    10.761      5.097      0.000
25%      14.731      6.815      0.522    14.695      6.763      0.509
50%      15.676      7.585      0.957    15.591      7.522      0.947
75%      16.564      8.165      1.374    16.430      8.109      1.351
max      19.394      9.176      2.164    19.417      9.176      2.164


## Walk-forward backtest: plain DC vs regression-DC

In [3]:
def score(y_true, proba):
    p = np.clip(proba, 1e-6, 1 - 1e-6)
    onehot = np.zeros_like(p); onehot[np.arange(len(y_true)), y_true] = 1
    return {
        'log_loss': log_loss(y_true, p, labels=[0,1,2]),
        'accuracy': float((p.argmax(axis=1) == y_true).mean()),
        'brier':    float(np.mean(np.sum((p - onehot)**2, axis=1))),
    }

rows = []
for year in [2010, 2014, 2018, 2022]:
    train = valid[valid['date'].dt.year < year]
    test  = valid[(valid['date'].dt.year == year) & (valid['tournament']=='FIFA World Cup')]
    if test.empty: continue
    y_test = test['outcome'].to_numpy()
    row = {'year': year, 'n_train': len(train), 'n_test': len(test)}
    
    # Plain DC
    dc = DixonColesModel(xi=0.00038, max_goals=10).fit(train)  # ~5y half-life
    preds_dc = dc.predict_many(test[['home_team','away_team','neutral']].fillna({'neutral': False}))
    p_dc = preds_dc[['p_away_win','p_draw','p_home_win']].to_numpy()
    s_dc = score(y_test, p_dc)
    row['dc_ll'], row['dc_acc'], row['dc_br'] = s_dc['log_loss'], s_dc['accuracy'], s_dc['brier']

    # Regression DC
    rdc = RegressionDixonColesModel(FEATURE_COLS, xi=0.00038).fit(train)
    preds_rdc = rdc.predict_many(test)
    p_rdc = preds_rdc[['p_away_win','p_draw','p_home_win']].to_numpy()
    s_rdc = score(y_test, p_rdc)
    row['rdc_ll'], row['rdc_acc'], row['rdc_br'] = s_rdc['log_loss'], s_rdc['accuracy'], s_rdc['brier']
    rows.append(row)

results = pd.DataFrame(rows)
print(results.round(4).to_string(index=False))
print()
print('Averages:')
for name in ['dc','rdc']:
    print(f"  {name:4s}  log_loss: {results[f'{name}_ll'].mean():.4f}   accuracy: {results[f'{name}_acc'].mean():.4f}   brier: {results[f'{name}_br'].mean():.4f}")

 year  n_train  n_test  dc_ll  dc_acc  dc_br  rdc_ll  rdc_acc  rdc_br
 2010     1488      64 1.0985  0.4688 0.6582  1.0206   0.5156  0.6127
 2014     2828      64 0.9830  0.5781 0.5870  1.0023   0.5938  0.6007
 2018     4195      64 0.9659  0.5938 0.5722  0.9970   0.5312  0.5953
 2022     5801      64 1.0333  0.5156 0.6036  1.0405   0.5312  0.6133

Averages:
  dc    log_loss: 1.0202   accuracy: 0.5391   brier: 0.6052
  rdc   log_loss: 1.0151   accuracy: 0.5430   brier: 0.6055


## Regression-DC coefficients

What did the model learn about each feature's contribution to attack and defense?

In [4]:
last_train = valid[valid['date'].dt.year < 2022]
rdc_final = RegressionDixonColesModel(FEATURE_COLS, xi=0.00038).fit(last_train)
print(rdc_final.coefficients().round(4).to_string(index=False))
print(f'\ngamma (log home adv): {rdc_final.home_adv:.4f}   exp(gamma)={np.exp(rdc_final.home_adv):.3f}x')
print(f'rho (low-score coupling): {rdc_final.rho:.4f}')

# Implied alpha/beta for top teams at end-of-2021 features
for team in ['Brazil','France','Argentina','Germany']:
    sub = last_train[(last_train['home_team']==team)|(last_train['away_team']==team)].tail(5)
    if sub.empty: continue
    # Use most recent feature values for this team
    is_home = sub['home_team']==team
    feats = []
    for c in FEATURE_COLS:
        v = sub.loc[is_home, f'home_{c}'].mean() if is_home.any() else sub[f'away_{c}'].mean()
        feats.append(v)
    feats = np.array(feats)
    x_c = feats - rdc_final.feature_mean_
    a = x_c @ rdc_final.w_a; b = x_c @ rdc_final.w_b
    print(f"{team:10s}  alpha={a:+.3f}  beta={b:+.3f}   (higher alpha=better attack, higher beta=better defense (subtracts more from opp goal rate))")

feature  w_attack  w_defense
    elo    0.1644     0.1594
   logv    0.1715     0.1948
   fpts    0.1720     0.1676

gamma (log home adv): 0.2656   exp(gamma)=1.304x
rho (low-score coupling): -0.0404
Brazil      alpha=+0.852  beta=+0.873   (higher alpha=better attack, higher beta=worse defense)
France      alpha=+0.856  beta=+0.879   (higher alpha=better attack, higher beta=worse defense)
Argentina   alpha=+0.796  beta=+0.815   (higher alpha=better attack, higher beta=worse defense)
Germany     alpha=+0.690  beta=+0.715   (higher alpha=better attack, higher beta=worse defense)
